In [4]:
# molecule preparation and minimization 

In [5]:
# Basic RDKit molecule preparation and minimization

In [6]:
from rdkit import Chem
from rdkit.Chem import AllChem

# Input SMILES
smiles = "CCO"  # ethanol example
mol = Chem.MolFromSmiles(smiles)

# Add hydrogens (important for 3D + force fields)
mol = Chem.AddHs(mol)

# Generate 3D coordinates
AllChem.EmbedMolecule(mol, AllChem.ETKDG())

# Energy minimization (UFF)
AllChem.UFFOptimizeMolecule(mol)

# Optionally remove hydrogens
mol_noH = Chem.RemoveHs(mol)

In [ ]:
# 3D conformation 

In [ ]:
import os
from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import AllChem
from rdkit.Chem.MolStandardize import rdMolStandardize

# 🔇 Disable RDKit spam logs
RDLogger.DisableLog('rdApp.*')

# ========= SETTINGS =========
MAX_TAUTOMERS = 10
KEEP_TOP_TAUTOMERS = 3
ALLOWED_CHARGES = [0, +1, -1]
# ============================


def get_user_paths():
    input_path = input("Enter input file path (.smi/.txt/.sdf): ").strip()
    output_path = input("Enter output SDF file path: ").strip()
    log_path = input("Enter log file path (e.g., failures.log): ").strip()

    if not os.path.exists(input_path):
        raise FileNotFoundError(f"Input file not found: {input_path}")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    if os.path.dirname(log_path):
        os.makedirs(os.path.dirname(log_path), exist_ok=True)

    return input_path, output_path, log_path


def safe_sanitize(mol):
    try:
        Chem.SanitizeMol(mol)
        return mol
    except Exception as e:
        return None


def load_molecules(path):
    ext = os.path.splitext(path)[1].lower()
    mols = []

    if ext in [".smi", ".txt"]:
        with open(path) as f:
            for i, line in enumerate(f):
                if not line.strip():
                    continue
                smi = line.split()[0]
                mol = Chem.MolFromSmiles(smi)
                if mol:
                    mol.SetProp("_InputID", str(i))
                    mols.append(mol)

    elif ext == ".sdf":
        for i, m in enumerate(Chem.SDMolSupplier(path, sanitize=False)):
            if m is None:
                continue
            m = safe_sanitize(m)
            if m:
                m.SetProp("_InputID", str(i))
                mols.append(m)

    else:
        raise ValueError("Supported formats: .smi, .txt, .sdf")

    return mols


def allowed_charge(mol):
    return Chem.GetFormalCharge(mol) in ALLOWED_CHARGES


def get_restricted_tautomers(mol):
    enumerator = rdMolStandardize.TautomerEnumerator()
    enumerator.SetMaxTautomers(MAX_TAUTOMERS)

    tautomers = list(enumerator.Enumerate(mol))
    canonical = enumerator.Canonicalize(mol)

    unique = []
    seen = set()

    for t in [canonical] + tautomers:
        smi = Chem.MolToSmiles(t)
        if smi not in seen:
            seen.add(smi)
            unique.append(t)

    return unique[:KEEP_TOP_TAUTOMERS]


def prepare_molecule(mol):
    mol = rdMolStandardize.Cleanup(mol)
    mol = rdMolStandardize.FragmentParent(mol)

    mol = rdMolStandardize.Uncharger().uncharge(mol)
    mol = rdMolStandardize.Reionizer().reionize(mol)

    if not allowed_charge(mol):
        raise ValueError("Disallowed charge state")

    tautomers = get_restricted_tautomers(mol)

    prepared = []

    for taut in tautomers:
        taut = Chem.AddHs(taut)

        # sanitize safely
        taut = safe_sanitize(taut)
        if taut is None:
            continue

        if AllChem.EmbedMolecule(taut, AllChem.ETKDG()) != 0:
            continue

        if AllChem.MMFFHasAllMoleculeParams(taut):
            AllChem.MMFFOptimizeMolecule(taut)
        else:
            AllChem.UFFOptimizeMolecule(taut)

        prepared.append(taut)

    if not prepared:
        raise ValueError("No valid tautomers generated")

    return prepared


# ===== MAIN =====
try:
    input_path, output_path, log_path = get_user_paths()

    molecules = load_molecules(input_path)
    print(f"Loaded {len(molecules)} molecules")

    writer = Chem.SDWriter(output_path)

    success = 0
    failed = 0

    with open(log_path, "w") as log_file:

        for mol in molecules:
            mol_id = mol.GetProp("_InputID") if mol.HasProp("_InputID") else "NA"

            try:
                prepared = prepare_molecule(mol)

                for j, pmol in enumerate(prepared):
                    pmol.SetProp("_Name", f"mol_{mol_id}_taut_{j}")
                    writer.write(pmol)

                success += 1

            except Exception as e:
                failed += 1

                try:
                    smi = Chem.MolToSmiles(mol)
                except:
                    smi = "N/A"

                log_file.write(f"ID: {mol_id} | SMILES: {smi} | Error: {str(e)}\n")

    writer.close()

    print(f"✅ Success: {success} molecules")
    print(f"❌ Failed: {failed} molecules")
    print(f"📄 Log file: {log_path}")
    print(f"💾 Output: {output_path}")

except Exception as e:
    print(f"Fatal Error: {e}")

Enter input file path (.smi/.txt/.sdf): C:/Users/ravit/Desktop/TYK2/TESTTYK2.sdf
Enter output SDF file path: C:/Users/ravit/Desktop/TYK2/TESTTYK2minus.sdf
Enter log file path (e.g., failures.log): C:/Users/ravit/Desktop/TYK2/TESTTYK2minlog.txt
Loaded 1842 molecules
